# 02 — Baseline Word2Vec + BiLSTM

Notebook ini melatih model baseline dengan konfigurasi final yang sudah
dipakai di paper. Tujuannya:

1. Memahami alur Word2Vec → BiLSTM.
2. Melihat cara membaca metrik dan kurva training.
3. Menyimpan artefak model yang bisa diekspor ke layanan inferensi.

Model ini menjadi pembanding untuk hasil HPO di notebook 03.

## Alur model

```
Teks komposisi
   │
   ▼
Tokenisasi (Keras Tokenizer)
   │
   ▼
Word2Vec 100 dimensi (dilatih hanya di final-train)
   │
   ▼
Embedding matrix
   │
   ▼
BiLSTM 128 → Dropout → BiLSTM 64 → Dropout
   │
   ▼
Dense 64 → Dropout → Sigmoid
   │
   ▼
Probabilitas unsafe
```

Keputusan `unsafe` jika probabilitas ≥ 0,5.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook biasanya dijalankan dari folder notebooks/. Kita pindah ke root
# project agar semua path artefak konsisten.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "artifacts" / "data"
BERT_DIR = PROJECT_ROOT / "artifacts" / "bert" / "input"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "notebook_runs"
for path in (DATA_DIR, BERT_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("BERT input   :", BERT_DIR)
print("Runs dir     :", RUNS_DIR)

In [ ]:
from app.experiment.configs import DataConfig
from app.experiment.data_pipeline import prepare_data

config_path = DATA_DIR / "data_config.json"
if config_path.exists():
    data_config = DataConfig.from_dict(json.loads(config_path.read_text(encoding="utf-8")))
else:
    candidates = [
        PROJECT_ROOT / "ocr_output" / "data-mengandung.csv",
        PROJECT_ROOT.parent / "old" / "ocr_output" / "data-mengandung.csv",
        PROJECT_ROOT / "data.csv",
    ]
    csv_path = next((path for path in candidates if path.exists()), None)
    if csv_path is None:
        raise FileNotFoundError(
            "CSV tidak ditemukan. Set DataConfig.csv_input secara manual "
            "ke file dengan kolom: nama_produk;text;label"
        )
    data_config = DataConfig(
        csv_input=str(csv_path),
        output_dir=str(DATA_DIR),
        bert_input_dir=str(BERT_DIR),
        sort_source=True,
    )

# prepare_data bersifat deterministik dan cepat; aman dijalankan ulang.
bundle = prepare_data(data_config)
print(bundle.summary_frame().to_string(index=False))

## Konfigurasi baseline

`QUICK_RUN = True` memakai 5 epoch agar notebook selesai cepat. Untuk hasil
paper, ubah ke `False` (20 epoch + early stopping).

In [ ]:
from app.experiment.configs import BiLSTMConfig

QUICK_RUN = True

config = BiLSTMConfig()
if QUICK_RUN:
    config.epochs = 5
    config.early_stopping_patience = 2

pd.DataFrame([config.to_dict()]).T.rename(columns={0: "nilai"})

> **Catatan perbandingan:** baseline baru memakai `shuffle=True` agar setara
> dengan IndoBERT yang selalu mengacak data latih. Konfigurasi legacy paper
> memakai `shuffle=False`; jangan mencampur angka dari kedua protokol tanpa
> menjelaskannya di laporan.

## Vektorisasi: Tokenizer + Word2Vec

Tokenizer dan Word2Vec **hanya dilatih pada final-train**. Validation dan
holdout hanya diubah menjadi angka memakai kosakata yang sudah ada. Ini
mencegah kebocoran kosakata.

In [ ]:
from app.experiment.bilstm_pipeline import build_vectorizers

vectorized = build_vectorizers(bundle, config)
print("Ukuran vocabulary tokenizer :", len(vectorized.tokenizer.word_index))
print("Ukuran vocabulary Word2Vec  :", len(vectorized.w2v_model.wv))
print("Bentuk embedding matrix     :", vectorized.embedding_matrix.shape)
print(f"Cakupan Word2Vec            : {vectorized.w2v_coverage:.1%}")
print("\nOOV per split:")
print(vectorized.oov_table.to_string(index=False))

## Latih model final baseline

Sel ini menjalankan training, evaluasi validation + holdout, dan menyimpan
artefak ke `artifacts/notebook_runs/bilstm_baseline/`.

Holdout hanya diprediksi **sekali** di akhir. Tidak ada early stopping dari
holdout.

In [ ]:
from app.experiment.bilstm_pipeline import train_final_bilstm

OUT_DIR = RUNS_DIR / "bilstm_baseline"
MODEL_DIR = OUT_DIR / "models"

output = train_final_bilstm(
    bundle,
    config,
    output_dir=OUT_DIR,
    model_dir=MODEL_DIR,
    hpo_info={"mode": "baseline_notebook"},
    verbose=1,
)
print("Selesai. Artefak:", OUT_DIR)

## Baca metrik

Metrik utama dihitung pada threshold 0,5:

- **Accuracy**: proporsi prediksi benar.
- **Precision**: dari yang diprediksi unsafe, berapa yang benar unsafe.
- **Recall**: dari yang benar unsafe, berapa yang berhasil ditemukan.
- **F1**: keseimbangan precision dan recall.
- **ROC-AUC**: kemampuan membedakan kelas di semua threshold.

In [ ]:
print(output["eval_table"].to_string(index=False))

In [ ]:
result = output["result"]
history = result.history

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["loss"], label="Train loss")
if "val_loss" in history:
    axes[0].plot(history["val_loss"], label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history["accuracy"], label="Train accuracy")
if "val_accuracy" in history:
    axes[1].plot(history["val_accuracy"], label="Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

for column in ("precision", "recall", "f1", "val_precision", "val_recall", "val_f1"):
    if column in history:
        axes[2].plot(history[column], label=column)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Score")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from app.experiment.reporting import plot_confusion, plot_roc_pr

plot_confusion(
    bundle.y_holdout, result.holdout_prob,
    OUT_DIR / "confusion_holdout.png",
    title="BiLSTM baseline — frozen holdout",
)
plot_roc_pr(
    {
        "Validation": (bundle.y_val, result.val_prob),
        "Frozen holdout": (bundle.y_holdout, result.holdout_prob),
    },
    OUT_DIR / "roc_pr.png",
    title="BiLSTM baseline",
)
print("Plot tersimpan di", OUT_DIR)

## Cara membaca hasil

- Jika **train accuracy naik** tetapi **validation accuracy turun**, model
  mulai menghafal (overfitting). Early stopping akan mengembalikan bobot
  terbaik.
- Jika **recall tinggi tetapi precision rendah**, model terlalu sering
  menebak `unsafe`. Untuk aplikasi alergi, recall tinggi sering lebih penting,
  tetapi false positive tetap mengganggu pengguna.
- Jika **ROC-AUC jauh di atas 0,5**, model punya kemampuan diskriminasi yang
  baik, meskipun accuracy bisa terlihat biasa saja.

Bandingkan angka ini dengan notebook 03 (setelah HPO) dan notebook 04/05
(IndoBERT). Gunakan holdout yang sama dan uji statistik di notebook 06.

## Masalah umum

| Gejala | Solusi |
|---|---|
| Loss tidak turun | Naikkan learning rate atau cek tokenisasi |
| Validation loss naik cepat | Turunkan epoch, naikkan dropout, pakai early stopping |
| Cakupan Word2Vec rendah | Samakan tokenisasi atau turunkan `w2v_min_count` |
| Hasil berubah-ubah | Set `PYTHONHASHSEED` sebelum Python, pakai seed sama |